# OpenAI-Compatible LLM API Connectivity
## OpenAI-Compatible Protocol — Two Paths: NVIDIA build · OpenRouter

> 📦 **Environment installation and run commands** are summarized in [`env_guides/M02_1_local_llm_eng.md`](env_guides/M02_1_local_llm_eng.md).
> Repeated/common code has been factored out into the [`agentic_lib/`](agentic_lib) library (bootstrap · tools).

---

### Prerequisites
- Windows 11 + **CMD (`cmd.exe`)** + Python **3.11** (managed by `uv`), kernel = **`Agentic AI (uv)`**
- Default LLM = **NVIDIA build + `deepseek-ai/deepseek-v4.1-flash`**, comparison target = **OpenRouter + `openrouter/free`**
- Set `NVIDIA_API_KEY` and `OPENROUTER_API_KEY` in `notebooks/.env` (see [`M02_0_free_llm_api_eng.ipynb`](M02_0_free_llm_api_eng.ipynb) for how to get them)
- For the one-time common setup (install uv → `uv venv` → `uv sync` → register kernel), see [`env_guides/README_eng.md`](env_guides/README_eng.md)

### Learning Objectives
This notebook runs the two cloud paths **each explicitly**, **regardless of** the `LLM_PROVIDER` setting in `.env`.
1. **Path 1 — NVIDIA build**: (1) connection test · (2) OpenAI-compatible response test
2. **Path 2 — OpenRouter**: (1) connection test · (2) OpenAI-compatible response test

Both services provide an **OpenAI-compatible `/v1`**, so we test them with **the same `ChatOpenAI` code**, changing only `base_url`, key, and model.

### Architecture Overview
```
                        ┌─→ [Path 1] NVIDIA build  · integrate.api.nvidia.com/v1 · deepseek-v4.1-flash
[LangChain ChatOpenAI] ─┤
                        └─→ [Path 2] OpenRouter    · openrouter.ai/api/v1        · openrouter/free
        (both are OpenAI-compatible /v1 — just change base_url and the code stays the same)
```

In [ ]:
# [setup] Self-contained setup — makes this notebook runnable on its own
import sys, os
sys.path.insert(0, os.path.abspath(''))   # add notebooks/ to the import path

import utils
utils.reload_env()   # reload .env (refresh LLM_PROVIDER, etc.) + print current provider status

from utils import get_llm, print_provider_status

# Repeated/common code is factored out into the agentic_lib library (this notebook uses only bootstrap)
from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text   # provider-agnostic response normalization (including <think> removal)

# Both paths are OpenAI-compatible → connect with a single ChatOpenAI from langchain-openai
utils.uv_install(['langchain', 'langchain-openai', 'langchain-nvidia-ai-endpoints',
                  'openai', 'requests'])

print_provider_status()

## 1. Environment Setup — OpenAI-Compatible Protocol

An **OpenAI-compatible API** is an API that follows the request/response format defined by OpenAI (`/v1/chat/completions`, `/v1/models`, etc.)
as-is. NVIDIA build and OpenRouter both provide this format, so by passing only `base_url`, `api_key`, and `model` to LangChain's `ChatOpenAI`,
you can switch between providers **without changing code**.

### How This Notebook Proceeds (independent of `.env`)
In §2 and §3 below, we run **the two paths each explicitly**. Whatever the `LLM_PROVIDER` value in `.env` is,
each path's `base_url`, key, and model are **specified directly in code**, so the result does not depend on that setting.

| Path | Provider | `base_url` | Model | Characteristics |
|---|---|---|---|---|
| **Path 1** | `nvidia` | `https://integrate.api.nvidia.com/v1` | `deepseek-ai/deepseek-v4.1-flash` | Free credits, fixed model |
| **Path 2** | `openrouter` | `https://openrouter.ai/api/v1` | `openrouter/free` | Automatic routing to free models |

For each path we perform **(1) a connection test** (OpenAI-compatible `/v1/models`) and **(2) a response test** (`ChatOpenAI.invoke`).

> For installation and troubleshooting, see [`env_guides/M02_1_local_llm_eng.md`](env_guides/M02_1_local_llm_eng.md).

In [ ]:
# [Common helpers] OpenAI-compatible API tests — (1) connection test · (2) protocol response test
# This notebook runs the two paths below 'each explicitly', regardless of LLM_PROVIDER in .env:
#   · Path 1: NVIDIA build — deepseek-ai/deepseek-v4.1-flash · integrate.api.nvidia.com/v1
#   · Path 2: OpenRouter   — openrouter/free                 · openrouter.ai/api/v1
# Both services are OpenAI-compatible /v1, so we test them with the same code, changing only base_url, key, and model.
import requests
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Connection info per path (uses keys and models read from .env)
ENDPOINTS = {
    "nvidia": dict(base_url=utils.NVIDIA_BASE_URL or "https://integrate.api.nvidia.com/v1",
                   api_key=utils.NVIDIA_API_KEY, model=utils.NVIDIA_MODEL),
    "openrouter": dict(base_url=utils.OPENROUTER_BASE_URL,
                       api_key=utils.OPENROUTER_API_KEY, model=utils.OPENROUTER_MODEL),
}

def check_openai_server(base_url, api_key, timeout=15.0):
    """Health-check via the OpenAI-compatible /v1/models and return the list of available models."""
    try:
        resp = requests.get(f"{base_url}/models", timeout=timeout,
                            headers={"Authorization": f"Bearer {api_key}"})
        resp.raise_for_status()
        return True, [m["id"] for m in resp.json().get("data", [])]
    except Exception as e:
        return False, str(e)

def connection_test(provider):
    """(1) 'Connection test' — check the server response and whether the configured model is offered, via OpenAI-compatible /v1/models."""
    ep = ENDPOINTS[provider]
    print(f"[{provider}] Connection test → {ep['base_url']}/models")
    if not ep["api_key"]:
        print(f"  ❌ Key not set — add {provider.upper()}_API_KEY to .env.")
        return False
    ok, info = check_openai_server(ep["base_url"], ep["api_key"])
    if not ok:
        print(f"  ❌ Connection failed: {info}")
        return False
    listed = ep["model"] in info
    print(f"  ✅ Server response OK — {len(info)} models available")
    print(f"  Configured model {ep['model']!r}: " + ("in the list ✅" if listed else "not in the list ⚠️ (check whether it has been retired)"))
    return ok

def make_openai_compatible(provider, temperature=0):
    """Create a ChatOpenAI with the path's base_url, key, and model — the code is the same regardless of provider."""
    ep = ENDPOINTS[provider]
    return ChatOpenAI(model=ep["model"], base_url=ep["base_url"], api_key=ep["api_key"],
                      temperature=temperature, max_tokens=1024, timeout=120)

def response_test(provider):
    """(2) 'OpenAI-compatible protocol response test' — run actual inference with ChatOpenAI.

    Because provider is passed directly as an argument, we connect to the desired path regardless of LLM_PROVIDER in .env.
    """
    llm = make_openai_compatible(provider)
    print(f"[{provider}] Response test · LLM type={type(llm).__name__} · base_url={ENDPOINTS[provider]['base_url']}")
    demo = llm.invoke([
        SystemMessage(content="You are an English-speaking assistant that answers concisely."),
        HumanMessage(content="Explain in one sentence the advantage of using an LLM through an OpenAI-compatible API."),
    ])
    # model_name in response_metadata: the model that actually generated the response (a router may differ per call)
    print(f"  Response model: {demo.response_metadata.get('model_name', '(no info)')}")
    print(f"  Response: {to_text(demo.content)}")      # to_text: provider-agnostic normalization (<think> removal, etc.)
    return llm

print("Helpers ready — connection_test(provider) · response_test(provider)")

---
## 2. Path 1 — NVIDIA build

The first path is **NVIDIA build** (build.nvidia.com). You can use a variety of models with free credits, and it
provides an **OpenAI-compatible endpoint** (`https://integrate.api.nvidia.com/v1`). The course's default model
**`deepseek-ai/deepseek-v4.1-flash`** responds quickly and can make **parallel calls** to multiple tools in a single response.

> In this notebook we connect with `ChatOpenAI` to demonstrate OpenAI compatibility. In other notebooks, `utils.get_llm('nvidia')` returns
> the NVIDIA-specific connector `ChatNVIDIA`, but both have the same `.invoke()` interface (see §4).

Steps: **(1) connection test** → **(2) OpenAI-compatible response test**

In [ ]:
# [Path 1] (1) NVIDIA build connection test — OpenAI-compatible /v1/models
connection_test("nvidia")

In [ ]:
# [Path 1] (2) NVIDIA build OpenAI-compatible protocol response test (deepseek-v4.1-flash)
nvidia_llm = response_test("nvidia")   # ChatOpenAI(base_url=integrate.api.nvidia.com/v1)

---
## 3. Path 2 — OpenRouter

The second path is **OpenRouter** (openrouter.ai). It is a **model router** that bundles models from many companies behind a single
**OpenAI-compatible endpoint** (`https://openrouter.ai/api/v1`).

The default model **`openrouter/free`** (Free Models Router) automatically picks an available free model for each request.
Pinning a specific `:free` model can fail due to overload (503) or rate limits (429), but the router falls over to another free model,
so it is more reliable. Note, however, that **the responding model may change on every call** (see the `Response model` output below).

> **Usage limits**: Free models are limited to 50 requests per day (1,000 requests after topping up 10 or more credits).

Steps: **(1) connection test** → **(2) OpenAI-compatible response test**

In [ ]:
# [Path 2] (1) OpenRouter connection test — OpenAI-compatible /v1/models
connection_test("openrouter")

In [ ]:
# [Path 2] (2) OpenRouter OpenAI-compatible protocol response test (openrouter/free)
openrouter_llm = response_test("openrouter")   # ChatOpenAI(base_url=openrouter.ai/api/v1)

---
## 4. Summary — Comparing the Two Paths (OpenAI-Compatible Protocol)

| Path | Service | `base_url` | Model | Characteristics |
|---|---|---|---|---|
| Path 1 | **NVIDIA build** | `integrate.api.nvidia.com/v1` | `deepseek-ai/deepseek-v4.1-flash` | Fixed model → reproducible results |
| Path 2 | **OpenRouter** | `openrouter.ai/api/v1` | `openrouter/free` | Automatic routing → availability |

Both paths were accessed with **the same `ChatOpenAI` code** (`make_openai_compatible(provider).invoke(...)`),
and the only differences are **`base_url`, key, and model**. This is the key benefit of the **OpenAI-compatible protocol**.
Even when using provider-specific connectors (such as `ChatNVIDIA`), the LangChain interface is the same,
so in other notebooks we switch providers with `utils.get_llm()` and a single `LLM_PROVIDER` line in `.env`.

In [ ]:
# [Summary] Check both paths side by side — connection status + whether the configured model is available
print("=== Two cloud LLM paths (OpenAI-compatible /v1) ===")
for prov in ["nvidia", "openrouter"]:
    ep = ENDPOINTS[prov]
    ok, info = check_openai_server(ep["base_url"], ep["api_key"])
    status = (f"✅ {len(info)} models available · configured model {'present' if ep['model'] in info else 'missing ⚠️'}"
              if ok else f"❌ {info}")
    print(f"  Path: {prov:11s} {ep['base_url']:38s} {ep['model']:34s} {status}")

# Check that the provider-specific connectors (utils.get_llm) share the same interface
print("\nConnectors returned by utils.get_llm():")
for prov in ["nvidia", "openrouter"]:
    print(f"  get_llm({prov!r:12s}) → {type(get_llm(prov)).__name__}  (.invoke / .stream / .bind_tools are the same)")